In [1]:
# chromブラウザのインストール

!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo 'deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main' | tee /etc/apt/sources.list.d/google-chrome.list
!apt-get update
!apt-get install google-chrome-stable
# !apt-get install fonts-ipaexfont-gothic fonts-ipaexfont-mincho                # 文字化け回避

OK
deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://archive.ubuntu.com/ubuntu noble InRelease
Get:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:5 http://dl.google.com/linux/chrome/deb stable InRelease [2,548 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:8 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:9 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,406 B]
Get:10 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,116 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,741 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-updates/restricted amd64 Packages [2,240 kB]
Get:13 http://archive.ubuntu.com/ubunt

In [2]:
!pip install requests selenium beautifulsoup4 webdriver-manager

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 74.4 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [selenium]


In [73]:
# インポート

from datetime import datetime, timezone, timedelta
from datetime import date
import time
import re
from bs4 import BeautifulSoup
from webdriver_manager.chrome import ChromeDriverManager
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains

In [74]:
# ヘッドレスモードで起動するためのオプションを設定
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

# ChromeDriverManagerを使用してChromeDriverを自動的に管理
service = Service(ChromeDriverManager().install())

# Google Chrome用のブラウザドライバーインスタンスを作成
chrome_driver = webdriver.Chrome(service=service, options=chrome_options)

In [75]:

def extract_stock_data(html, formatted_today, is_first, previous_date):

    soup = BeautifulSoup(html, 'html.parser')
    legend = soup.find('div', class_='modcharts-legend')

    datas = []

    # 日付を変換してリストの先頭に追加
    date_text = legend.find('div', class_='chartworks-legend-timestamp').text
    date = datetime.strptime(date_text, '%Y年%m月%d日').date().strftime('%Y-%m-%d')


    # 最初にリストにセットするのは、今日の日付
    if is_first:
        if date == formatted_today:
            is_first = False
            previous_date = date

            datas = [date]
            for key in ['o', 'h', 'l', 'c']:
                # 始値（o）、高値（h）、安値（l）、終値（c）を抽出してリストに追加
                value = legend.find('span', class_=f'chartworks-legend-ohlc-{key}')
                datas.append(re.sub(r'\D', '', value.text))
    else:

    # リストの２つ目以降は、前回セットの日付と比較しながらセット
        if date != previous_date:
            previous_date = date

            datas = [date]
            for key in ['o', 'h', 'l', 'c']:
                # 始値（o）、高値（h）、安値（l）、終値（c）を抽出してリストに追加
                value = legend.find('span', class_=f'chartworks-legend-ohlc-{key}')
                datas.append(re.sub(r'\D', '', value.text))

    return datas, is_first, previous_date


def get_stock_values(driver, url):

    driver.get(url)

    # Waitオブジェクトを生成
    wait = WebDriverWait(chrome_driver, 30)

    # チャート本体が表示されるまで待つ
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '.chartworks-legend-timestamp')))

    # グラフの要素を取得する
    graph_element = driver.find_element(By.CSS_SELECTOR, '.modcharts-panel-root')

    # グラフ要素のサイズ（幅）を取得する
    graph_size = graph_element.size
    graph_width = graph_size['width']
    graph_height = graph_size['height']

    # 中央から右端までの移動距離を計算する
    x_offset = int(graph_width / 2)
    y_offset = 0  # Y軸は中央のまま（上下に動かさない）

    # ActionChainsを使って右端まで移動を実行
    actions = ActionChains(chrome_driver)

    actions.move_to_element(graph_element)
    actions.move_by_offset(x_offset, y_offset)
    actions.perform()
    time.sleep(1)

    # ストック用リスト初期化
    stock_datas = []

    # １件目の日付を取得
    JST = timezone(timedelta(hours=9))
    now_jst = datetime.now(JST)
    today = now_jst.date()

    if now_jst.hour < 9:
        today = today - timedelta(days=1)
    formatted_today = today.strftime("%Y-%m-%d")


    # １件目用フラグ
    is_first = True

    # 前回取得した日付
    previous_date = None

    # データの取得
    for x in range(graph_width, 0, -1):

        html = chrome_driver.page_source.encode('utf-8')
        result_data, is_first, previous_date = extract_stock_data(html, formatted_today, is_first, previous_date)

        if result_data != []:
            stock_datas.append(result_data)

        actions.move_by_offset(-1,0)                   # マウスを右に移動させる処理
        actions.perform()
        time.sleep(1)

    return stock_datas



#
start_time = datetime.now()

url = 'https://www.nikkei.com/smartchart/?code=N101%2FT&timeframe=6m&interval=1Day&upperIndicators=sma&lowerIndicators=volume&eventsShow=1'

# 株価データ取得
result = get_stock_values(chrome_driver, url)

# スクレイピングの終了時間
end_time = datetime.now()
elapsed_time = end_time - start_time
print(f"スクレイピングにかかった時間: {elapsed_time}")

# 閉じる
chrome_driver.quit()

# 取得データの表示
print("  日 付    始値  高値  安値  終値")
for row in result:
    print(*row)

スクレイピングにかかった時間: 0:08:41.697014
  日 付    始値  高値  安値  終値
2026-11-04 62973 63105 60030 62189
2026-11-02 62972 63615 62713 63082
2026-10-30 64344 64589 62711 63290
2026-10-29 64126 64831 63749 64659
2026-10-28 63739 64309 63660 63954
2026-10-27 62569 63685 62389 63268
2026-10-26 62450 62903 62094 62190
2026-10-23 62667 62787 61463 62693
2026-10-22 64539 64866 62908 62908
2026-10-21 64886 65405 64559 64621
2026-10-20 64755 64822 63807 64075
2026-10-19 64820 65068 63832 64917
2026-10-16 64925 65712 64801 65324
2026-10-15 66880 66929 64093 64426
2026-10-14 67942 68408 67533 67744
2026-10-13 68565 68711 67059 67778
2026-10-09 68649 69143 68150 69031
2026-10-08 69841 69918 69042 69042
2026-10-07 70582 70793 70017 70036
2026-10-06 69955 70799 69813 70684
2026-10-05 69114 70072 69050 69947
2026-10-02 68313 68741 68132 68309
2026-10-01 67107 68995 67082 68957
2026-09-30 66129 66946 66014 66754
2026-09-29 65558 65805 64699 65481
2026-09-28 66506 67035 65878 65878
2026-09-25 65640 66410 65640 66364
